# Exercice 30a (cloud) : d'un complèteur de texte à un assistant, par réglage fin complet, LoRA et QLoRA

Un modèle **pré-entraîné** (ici `HuggingFaceTB/SmolLM2-360M`, 360 millions de paramètres, licence
Apache-2.0) ne fait que prolonger un texte. Donnez-lui une question, il peut écrire la question
suivante d'un QCM, une suite de forum ou une réponse qui ne s'arrête jamais. Le **réglage par
instructions** (SFT, *supervised fine-tuning*) lui montre quelques milliers d'exemples
« instruction → réponse » et lui apprend le format : répondre, puis s'arrêter.

On fait le même SFT de trois façons, avec le même nombre de pas, et l'on compare mémoire, durée
et perte :

1. **Réglage fin complet** : tous les poids bougent (362 M paramètres, avec leurs gradients et les deux états d'Adam).
2. **LoRA** (Hu et al., 2021) : poids gelés en float16, on n'entraîne que des matrices de rang 16
   ajoutées à chaque projection linéaire, $W x + \frac{\alpha}{r} B A x$.
3. **QLoRA** (Dettmers et al., 2023) : la même chose, mais les poids gelés sont stockés sur
   **4 bits** (format NF4) et décompressés à la volée.

Avant cela, la cellule 4 écrit une couche LoRA **à la main** en PyTorch, pour voir ce que fait la
bibliothèque PEFT.

**Perte optimisée** : l'entropie croisée du token suivant, calculée **sur les tokens de la
réponse seulement** (ceux de l'instruction sont masqués) :
$\mathcal L = -\frac{1}{|y|}\sum_t \log p_\theta(y_t \mid x, y_{<t})$.

Avant de lancer : *Exécution › Modifier le type d'exécution › GPU T4*, puis *Exécution › Tout
exécuter*. Seed 42. Calcul en **float16** (le T4 ne fait pas de bfloat16). Aucun token Hugging
Face n'est nécessaire.

In [ ]:
# Cellule 1 : vérifier le GPU
!nvidia-smi
import os
# Kaggle (« GPU T4 x2 ») : TRL refuse de tourner avec deux GPU visibles sans lanceur distribué.
# On n'en garde qu'un, avant d'importer torch. Sur Colab, cette ligne ne fait rien.
if os.path.exists("/kaggle/working") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import torch
TEST_LOCAL = os.environ.get("IA_30A_TEST") == "1"   # vérification du code en miniature, hors GPU (README)
if not TEST_LOCAL and not torch.cuda.is_available():
    raise SystemExit("Pas de GPU : Exécution › Modifier le type d'exécution › GPU T4, puis relancer.")
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# Cellule 2 : dépendances épinglées (torch n'est PAS épinglé : on garde celui de l'image Colab)
!pip install -q transformers==5.19.0 trl==1.15.0 peft==0.21.2 accelerate==1.15.0 datasets==5.1.0 bitsandbytes==0.50.2

In [ ]:
# Cellule 3 : configuration
import gc, json, math, random, time
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# COURT = True : 30 pas par variante, pour vérifier que tout tourne (≈ 5 min, estimation non mesurée)
COURT = False
config = dict(
    modele="HuggingFaceTB/SmolLM2-360M",           # modèle de BASE, Apache-2.0, non restreint (gated: false)
    donnees="databricks/databricks-dolly-15k",     # CC BY-SA 3.0, 15 011 paires écrites par des humains
    n_train=2000, n_eval=200,
    max_length=512,                                # instruction + réponse, en tokens
    n_pas=30 if COURT else 250,                    # même nombre de pas pour les trois variantes
    lot=4, accumulation=2,                         # lot effectif : 8 exemples
    lr_complet=2e-5, lr_lora=2e-4,                 # LoRA supporte (et demande) un taux bien plus fort
    rang=16, alpha=32,                             # LoRA : échelle α/r = 2
    nouveaux_tokens=120,
    variantes=["complet", "lora", "qlora"],
)
dispositif = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
if TEST_LOCAL:   # miniature : quelques pas, pas de float16 ni de 4 bits (ne vérifie que le code)
    config.update(n_train=48, n_eval=16, max_length=256, n_pas=4, lot=2, nouveaux_tokens=30,
                  variantes=["complet", "lora"] + (["qlora"] if dispositif == "cuda" else []))
FP16 = dispositif == "cuda"
print(config, dispositif, "float16" if FP16 else "float32")

In [ ]:
# Cellule 4 : une couche LoRA à la main (CPU, quelques secondes)
# W est gelée ; on apprend B (d_sortie × r) et A (r × d_entrée). B part de zéro :
# au départ, la couche fait exactement la même chose que l'originale.
from torch import nn

class LoRALineaire(nn.Module):
    def __init__(self, couche: nn.Linear, r: int, alpha: float):
        super().__init__()
        self.couche = couche.requires_grad_(False)                       # poids d'origine, gelés
        self.A = nn.Parameter(torch.randn(r, couche.in_features) / math.sqrt(couche.in_features))
        self.B = nn.Parameter(torch.zeros(couche.out_features, r))
        self.echelle = alpha / r

    def forward(self, x):
        return self.couche(x) + self.echelle * (x @ self.A.T) @ self.B.T   # W x + (α/r)·B A x

torch.manual_seed(SEED)
d = 960                                      # largeur des couches de SmolLM2-360M (hidden_size)
W = nn.Linear(d, d, bias=False)
x = torch.randn(256, d)
lora = LoRALineaire(W, r=16, alpha=32)
print("Sortie identique au départ :", torch.allclose(lora(x), W(x)))
print(f"Paramètres : W = {d * d:,} gelés ; A et B = {lora.A.numel() + lora.B.numel():,} entraînés "
      f"({(lora.A.numel() + lora.B.numel()) / (d * d):.1%})".replace(",", " "))

# Cible : la même couche après une « adaptation » de rang 4 (ΔW = U V, inconnue de l'élève)
U, V = torch.randn(d, 4) / 8, torch.randn(4, d) / 8
cible = lambda x: W(x) + x @ (U @ V).T
for r in [1, 2, 4, 16]:
    torch.manual_seed(SEED)
    l = LoRALineaire(W, r=r, alpha=r)
    opt = torch.optim.Adam([l.A, l.B], lr=1e-2)
    for pas in range(300):
        xb = torch.randn(256, d)
        perte = ((l(xb) - cible(xb)) ** 2).mean()
        opt.zero_grad(); perte.backward(); opt.step()
    rel = (perte / (cible(xb) - W(xb)).pow(2).mean()).item()
    print(f"rang {r:2d} : erreur résiduelle {rel:.1%} de la correction à apprendre")

In [ ]:
# Cellule 5 : charger le modèle de base et le jeu d'instructions
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(config["modele"])
if tok.pad_token is None:          # le modèle de base n'a pas de token de remplissage
    tok.pad_token = tok.eos_token
print("fin de texte :", repr(tok.eos_token))
FORMAT = "### Instruction:\n{instruction}\n\n### Response:\n"     # gabarit de prompt (texte brut)

ds = load_dataset(config["donnees"], split="train")
# Questions sans texte d'appui (pas de champ « context ») et réponses de longueur raisonnable
ds = ds.filter(lambda e: e["context"] == "" and 20 <= len(e["response"]) <= 1200)
ds = ds.shuffle(seed=SEED)
print(ds, "\nCatégories :", sorted(set(ds["category"])))

def vers_prompt_completion(e):
    # Format « prompt / completion » de TRL : la perte ne porte que sur la completion
    return {"prompt": FORMAT.format(instruction=e["instruction"].strip()),
            "completion": e["response"].strip()}

donnees = ds.map(vers_prompt_completion, remove_columns=ds.column_names)
train = donnees.select(range(config["n_train"]))
evaluation = donnees.select(range(config["n_train"], config["n_train"] + config["n_eval"]))
print(train[0]["prompt"] + train[0]["completion"])

QUESTIONS = [
    "Give three tips for staying focused while working from home.",
    "What is the difference between a list and a tuple in Python?",
    "Écris une courte phrase pour souhaiter un bon anniversaire à une collègue.",
]

@torch.no_grad()
def repondre(modele, questions=QUESTIONS):
    modele.eval()
    sorties = []
    for q in questions:
        entree = tok(FORMAT.format(instruction=q), return_tensors="pt").to(modele.device)
        gen = modele.generate(**entree, max_new_tokens=config["nouveaux_tokens"], do_sample=False,
                              pad_token_id=tok.pad_token_id)
        nouveaux = gen[0, entree["input_ids"].shape[1]:]
        sorties.append(dict(question=q, reponse=tok.decode(nouveaux, skip_special_tokens=True),
                            n_tokens=int(len(nouveaux)),
                            arret=bool(tok.eos_token_id in nouveaux.tolist())))
    return sorties

def afficher(sorties, titre):
    print(f"===== {titre}")
    for s in sorties:
        fin = "s'arrête seul" if s["arret"] else f"coupé à {s['n_tokens']} tokens"
        print(f"--- {s['question']}\n{s['reponse'].strip()}\n[{fin}]\n")

In [ ]:
# Cellule 6 : AVANT, le modèle de base répond-il ?
dtype_base = torch.float16 if FP16 else torch.float32
base = AutoModelForCausalLM.from_pretrained(config["modele"], dtype=dtype_base).to(dispositif)
n_params = sum(p.numel() for p in base.parameters())
print(f"{n_params / 1e6:.0f} M de paramètres")
avant = repondre(base)
afficher(avant, "modèle de base, avant SFT")
del base; gc.collect()
if dispositif == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# Cellule 7 : une fonction qui fait le SFT selon la variante, et mesure tout
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig, TrainerCallback
from trl import SFTConfig, SFTTrainer

class Journal(TrainerCallback):
    # garde les pertes d'entraînement et d'évaluation au fil des pas
    def __init__(self):
        self.train, self.eval = [], []
    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs:
            self.train.append((state.global_step, logs["loss"]))
        if logs and "eval_loss" in logs:
            self.eval.append((state.global_step, logs["eval_loss"]))

def charger(variante):
    if variante == "complet":
        # poids maîtres en float32 ; le calcul passe en float16 par autocast (fp16=True)
        return AutoModelForCausalLM.from_pretrained(config["modele"], dtype=torch.float32)
    if variante == "lora":
        return AutoModelForCausalLM.from_pretrained(config["modele"], dtype=dtype_base)
    # qlora : poids gelés stockés en NF4 (4 bits), calcul en float16
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    m = AutoModelForCausalLM.from_pretrained(config["modele"], quantization_config=q, dtype=torch.float16)
    return prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)

def sft(variante):
    torch.manual_seed(SEED)
    if dispositif == "cuda":
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    modele = charger(variante)
    if variante != "qlora":
        modele = modele.to(dispositif)
    memoire_poids = torch.cuda.memory_allocated() / 2**30 if dispositif == "cuda" else float("nan")
    peft_config = None
    if variante != "complet":
        # LoRA sur toutes les projections linéaires des blocs (attention et MLP)
        peft_config = LoraConfig(r=config["rang"], lora_alpha=config["alpha"], lora_dropout=0.05,
                                 target_modules="all-linear", task_type="CAUSAL_LM")
    args = SFTConfig(
        output_dir=f"sortie-{variante}", seed=SEED, report_to="none",
        max_steps=config["n_pas"], per_device_train_batch_size=config["lot"],
        gradient_accumulation_steps=config["accumulation"], per_device_eval_batch_size=8,
        learning_rate=config["lr_complet"] if variante == "complet" else config["lr_lora"],
        lr_scheduler_type="cosine", warmup_steps=max(1, config["n_pas"] // 20),
        fp16=FP16, gradient_checkpointing=True, max_length=config["max_length"],
        completion_only_loss=True,                  # perte sur la réponse seulement
        logging_steps=max(1, config["n_pas"] // 25),
        eval_strategy="steps", eval_steps=max(1, config["n_pas"] // 5),
        save_strategy="no",
        optim="adamw_torch",
    )
    journal = Journal()
    entraineur = SFTTrainer(model=modele, args=args, train_dataset=train, eval_dataset=evaluation,
                            processing_class=tok, peft_config=peft_config, callbacks=[journal])
    m = entraineur.model
    n_entraines = sum(p.numel() for p in m.parameters() if p.requires_grad)
    perte_avant = entraineur.evaluate()["eval_loss"]
    t0 = time.perf_counter()
    entraineur.train()
    duree = time.perf_counter() - t0
    perte_apres = entraineur.evaluate()["eval_loss"]
    pic = torch.cuda.max_memory_allocated() / 2**30 if dispositif == "cuda" else float("nan")
    reponses = repondre(m)
    if variante == "lora":
        m.save_pretrained("adaptateur-lora")       # quelques dizaines de Mo, pas 1,2 Go
    resultat = dict(variante=variante, parametres_entraines=n_entraines,
                    part_entrainee=n_entraines / n_params, memoire_poids_gio=memoire_poids,
                    memoire_pic_gio=pic, duree_s=duree, s_par_pas=duree / config["n_pas"],
                    perte_eval_avant=perte_avant, perte_eval_apres=perte_apres,
                    courbe_train=journal.train, courbe_eval=journal.eval, reponses=reponses)
    del entraineur, m, modele; gc.collect()
    if dispositif == "cuda":
        torch.cuda.empty_cache()
    print(f"{variante}: {n_entraines / 1e6:.1f} M paramètres entraînés ({n_entraines / n_params:.2%}), "
          f"pic mémoire {pic:.2f} Gio, {duree / 60:.1f} min, perte d'évaluation {perte_avant:.3f} → {perte_apres:.3f}")
    return resultat

In [ ]:
# Cellule 8 : les trois entraînements (le plus long de l'exercice)
resultats = {}
for v in config["variantes"]:
    print(f"\n######## {v}")
    resultats[v] = sft(v)
    afficher(resultats[v]["reponses"], f"après SFT ({v})")

In [ ]:
# Cellule 9 : tableau comparatif et courbes
import matplotlib.pyplot as plt

print(f"{'variante':9s} {'entraînés':>12s} {'mémoire pic':>12s} {'s/pas':>7s} {'perte éval':>16s}")
for v, r in resultats.items():
    print(f"{v:9s} {r['parametres_entraines'] / 1e6:9.1f} M  {r['memoire_pic_gio']:9.2f} Gio "
          f"{r['s_par_pas']:7.2f}   {r['perte_eval_avant']:.3f} → {r['perte_eval_apres']:.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
for v, r in resultats.items():
    if r["courbe_eval"]:
        # dict : l'évaluation finale peut être journalisée deux fois au même pas
        pas, pertes = zip(*sorted(dict([(0, r["perte_eval_avant"])] + r["courbe_eval"]).items()))
        ax.plot(pas, pertes, "o-", label=v)
ax.set(xlabel="pas d'optimisation (lots de 8)", ylabel="perte d'évaluation (réponses seulement)",
       title="SFT de SmolLM2-360M : trois façons")
ax.legend(); fig.tight_layout(); fig.savefig("30a-pertes.png", dpi=120); plt.show()

with open("30a-resultats.json", "w") as f:
    json.dump(dict(config=config, avant=avant, resultats=resultats), f, ensure_ascii=False, indent=1)

In [ ]:
# Cellule 10 : récupérer les résultats avant la fin de session (Colab ; sur Kaggle : onglet Output)
import shutil
fichiers = ["30a-resultats.json", "30a-pertes.png"]
if os.path.isdir("adaptateur-lora"):
    shutil.make_archive("adaptateur-lora", "zip", ".", "adaptateur-lora"); fichiers.append("adaptateur-lora.zip")
print(fichiers)
try:
    from google.colab import files
    for f in fichiers:
        files.download(f)
except ImportError:
    print("Hors Colab : fichiers laissés dans le dossier courant.")